<h1 align=center style="line-height:200%">
<font color="#0099cc">
تحلیل جست‌وجوی کاربران مستربلیط
</font>
</h1>

<p dir="rtl" align="right">
تحلیل داده‌های جست‌وجوی کاربران در وب‌سایت رزرو سفر <b>مستربلیط</b> برای پاسخ به چند پرسش کلیدی کسب‌وکاری: کدام سرویس‌ها محبوب‌ترند؟ کاربران بیشتر به کدام شهرها و استان‌ها سفر می‌کنند؟ و کدام شهرهای پرجمعیت هنوز به‌اندازه‌ی کافی دیده نشده‌اند؟
</p>

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio

pd.set_option('display.max_columns', None)

In [ ]:
data = pd.read_json('../data/mrbilit_search.json')
data.head()

In [ ]:
cities = pd.read_csv('../data/iran_cities.csv')
cities.head()

In [ ]:
print('shape:', data.shape)
data.info()

In [ ]:
data.isnull().sum()

In [ ]:
# اطمینان از این‌که ServiceType دقیقاً همین ۶ مقدار را دارد؛ هر مقدار غیرمنتظره اینجا خودش را نشان می‌دهد
data['ServiceType'].unique()

In [ ]:
percentages = data['ServiceType'].value_counts(normalize=True).round(4).to_dict()
percentages

In [ ]:
fig = px.pie(values=percentages.values(),
             names=list(percentages.keys()),
             title='درصد محبوبیت سرویس‌ها',
             color_discrete_sequence=px.colors.qualitative.Set2)
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.write_image('../images/service_popularity.png', scale=2)
fig.show()

In [ ]:
def extract_city(value: str) -> str:
    """Keeps only the city part of an AcceptString value (before ' - ' if present)."""
    return value.split('-')[0].strip() if '-' in value else value.strip()


preprocessed_data = data.copy()
preprocessed_data['AcceptString'] = preprocessed_data['AcceptString'].apply(extract_city)
preprocessed_data.head()

In [ ]:
# پرجست‌وجوترین شهرها

data_transport = preprocessed_data[preprocessed_data['ServiceType'] != 'hotel']
top_cities_counts = data_transport['AcceptString'].value_counts().head(20)
top_cities = list(top_cities_counts.index)
top_cities

In [ ]:
fig = px.bar(
    x=top_cities_counts.values,
    y=top_cities_counts.index,
    orientation='h',
    title='20هر پرجست‌وجو (به تفکیک سرویس‌های حمل‌ونقل)',
    labels={'x': 'تعداد جست‌وجو', 'y': 'شهر'},
    color=top_cities_counts.values,
    color_continuous_scale='Blues',
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, coloraxis_showscale=False)
fig.write_image('../images/top_cities.png', scale=2)
fig.show()

In [ ]:
# پرجست‌وجوترین استان‌ها

merged = pd.merge(data_transport, cities, how='inner', left_on='AcceptString', right_on='City FA')

province_counts = merged['Province'].value_counts().head(15)
top_provinces = list(province_counts.index)
top_provinces

In [ ]:
fig = px.bar(
    x=province_counts.values,
    y=province_counts.index,
    orientation='h',
    title='۱۵ استان پرجست‌وجو',
    labels={'x': 'تعداد جست‌وجو', 'y': 'استان'},
    color=province_counts.values,
    color_continuous_scale='Oranges',
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, coloraxis_showscale=False)
fig.write_image('../images/top_provinces.png', scale=2)
fig.show()

In [ ]:
# نمایش جغرافیایی شهرهای پرجست‌وجو (تحلیل تکمیلی)

top_cities_geo = cities[cities['City FA'].isin(top_cities)].copy()
top_cities_geo['SearchCount'] = top_cities_geo['City FA'].map(top_cities_counts)

# نمایش به‌شکل نمودار حبابی روی مختصات جغرافیایی (بدون نیاز به نقشه‌ی پس‌زمینه‌ی آنلاین)
fig = px.scatter(
    top_cities_geo,
    x='Longitude',
    y='Latitude',
    size='SearchCount',
    color='SearchCount',
    text='City FA',
    hover_name='City FA',
    hover_data={'Province': True, 'SearchCount': True, 'Longitude': False, 'Latitude': False},
    color_continuous_scale='Plasma',
    size_max=45,
    title='پراکندگی جغرافیایی ۲۰ شهر پرجست‌وجو (طول و عرض جغرافیایی)',
)
fig.update_traces(textposition='top center', textfont_size=10)
fig.update_layout(
    xaxis_title='طول جغرافیایی',
    yaxis_title='عرض جغرافیایی',
    yaxis_scaleanchor='x',
    plot_bgcolor='#eef6fb',
)
fig.write_image('../images/cities_map.png', scale=2)
fig.show()

In [ ]:
# شهرهای پرجمعیت اما کم‌جست‌وجو (فرصت بازار)

census_500 = cities[cities['2016 Census'] > 500000]
not_in_top = [city for city in census_500['City FA'] if city not in top_cities]
not_in_top

In [ ]:
underserved = cities[cities['City FA'].isin(not_in_top)][['City FA', 'Province', '2016 Census']]
underserved = underserved.sort_values('2016 Census', ascending=False).reset_index(drop=True)
underserved

In [ ]:
fig = px.bar(
    underserved,
    x='2016 Census',
    y='City FA',
    orientation='h',
    color='Province',
    title='شهرهای پرجمعیت (۵۰۰هزار+) خارج از ۲۰ شهر پرجست‌وجو',
    labels={'2016 Census': 'جمعیت (سرشماری ۲۰۱۶)', 'City FA': 'شهر'},
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.write_image('../images/underserved_cities.png', scale=2)
fig.show()

**یافته و پیشنهاد بازاریابی:** شهرهایی مانند اردبیل، همدان و ارومیه با وجود جمعیت قابل‌توجه، هنوز جزو ۲۰ شهر پرجست‌وجوی پلتفرم نیستند. این شهرها می‌توانند هدف کمپین‌های تبلیغاتی محلی یا تخفیف‌های ورود به بازار (market-entry discounts) قرار بگیرند تا سهم مستربلیط در این مناطق افزایش یابد.